In [0]:
# ============================================================
# CELL 1 — Gold Layer: Generate Predictions + Business Output
# Notebook: 05_gold_layer
# Model: Ridge Regression (production winner)
# Output: gold_predictions Delta table with RUL scores,
#         alert classifications, and cost-avoidance framing
# ============================================================

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType
from sklearn.metrics import mean_squared_error, mean_absolute_error

EXPERIMENT_PATH = "/Users/gkolthoff1@live.maryville.edu/rul_prediction_cmapss"
ALERT_THRESHOLD = 30        # cycles — failure window
COST_PER_HOUR   = 10000     # conservative estimate: $10K/hour unplanned downtime
CYCLES_PER_HOUR = 1         # 1 engine cycle ≈ 1 flight hour (CMAPSS convention)

# ── Load Silver tables (full — train + test) ──
silver_fd001 = spark.table("workspace.predictive_maintenance.silver_fd001")
silver_fd003 = spark.table("workspace.predictive_maintenance.silver_fd003")

# ── Feature columns (must match modeling notebook exactly) ──
feature_cols = (
    [f"sensor_{s}_norm" for s in [2,3,4,7,8,9,11,12,13,15,17,20,21]] +
    [f"sensor_{s}_roll_mean" for s in [2,3,4,7,8,9,11,12,13,15,17,20,21]] +
    [f"sensor_{s}_roll_std" for s in [2,3,4,7,8,9,11,12,13,15,17,20,21]]
)

# ── Load the registered Ridge model from MLflow ──
mlflow.set_experiment(EXPERIMENT_PATH)

# Get the best Ridge run by test_rmse
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_PATH)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="params.model_type = 'RidgeCV'",
    order_by=["metrics.test_rmse ASC"],
    max_results=1
)

if not runs:
    raise ValueError("No Ridge run found in MLflow. Re-run Cell 2 of 04_modeling first.")

best_run = runs[0]
run_id   = best_run.info.run_id
print(f"Loading Ridge model from run: {run_id}")
print(f"  Test RMSE: {best_run.data.metrics.get('test_rmse', 'N/A'):.2f}")
print(f"  Best alpha: {best_run.data.params.get('best_alpha', 'N/A')}")

ridge_model = mlflow.sklearn.load_model(f"runs:/{run_id}/ridge_model")
print("✅ Model loaded successfully")

Loading Ridge model from run: 9143258b4b414e749cbb47bae61eab52
  Test RMSE: 23.88
  Best alpha: 100.0


✅ Model loaded successfully


{"ts": "2026-08-26 23:41:52.017", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDI4NDEwNDIzNDU4NzYxNhABIAEyJDAxYTA0MDczLTQ2NDgtNzExNC04OWY3LTZmMTBjOWE1MWFiYTokOTVkYzA0ZjYtZTI3NS0zMjFiLWEwOWYtZmE1YTExNzk2MmIwSgwIq/O91AYQgOikmgJQAVgBYAFosLmLqOrEow0=.", "context": {}}
{"ts": "2026-08-26 23:41:52.017", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDI4NDEwNDIzNDU4NzYxNhABIAEyJDAxYTA0MDczLTQ2NDgtNzExNC04OWY3LTZmMTBjOWE1MWFiYTokOTVkYzA0ZjYtZTI3NS0zMjFiLWEwOWYtZmE1YTExNzk2MmIwSgwIq/O91AYQgOikmgJQAVgBYAFosLmLqOrEow0=.", "context": {}}
{"ts": "2026-08-26 23:41:52.017", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDI4NDEwNDIzNDU4NzYxNhABIAEyJDAxYTA0MDczLTQ2NDgtNzExNC04OWY3LTZmMTBjOWE1MWFiYTokOTVkYzA0ZjYtZTI3NS0zMjFiLWEwOWYtZmE1YTExNzk2MmIw

In [0]:
# ============================================================
# CELL 2 — Generate Gold Predictions + Write Delta Table
# ============================================================

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType
from datetime import datetime

def generate_gold(silver_df, subset_label):
    """
    Apply Ridge model to Silver features, add business framing,
    return Gold DataFrame ready for Delta write.
    """
    # Convert to pandas for sklearn inference
    pdf = silver_df.toPandas()
    X   = pdf[feature_cols].fillna(0)

    # Predict and clip to valid RUL range
    rul_pred = np.clip(ridge_model.predict(X), 0, 125)

    # Business columns
    alert_30       = (rul_pred <= ALERT_THRESHOLD).astype(int)
    alert_urgency  = pd.cut(
        rul_pred,
        bins=[-1, 10, 30, 60, 125],
        labels=["CRITICAL", "WARNING", "MONITOR", "HEALTHY"]
    ).astype(str)
    cost_at_risk   = np.where(
        alert_30 == 1,
        (ALERT_THRESHOLD - rul_pred).clip(0) * COST_PER_HOUR * CYCLES_PER_HOUR,
        0.0
    )

    pdf["rul_predicted"]    = rul_pred.round(2)
    pdf["rul_actual"]       = pdf["RUL_capped"]
    pdf["rul_error"]        = (rul_pred - pdf["RUL_capped"]).round(2)
    pdf["alert_30"]         = alert_30
    pdf["alert_urgency"]    = alert_urgency
    pdf["cost_at_risk_usd"] = cost_at_risk.round(2)
    pdf["model_version"]    = "ridge_cv_alpha100"
    pdf["predicted_at"]     = datetime.utcnow().isoformat()
    pdf["gold_subset"]      = subset_label

    # Select Gold columns only
    gold_cols = [
        "unit_id", "time_cycles", "split", "subset",
        "rul_actual", "rul_predicted", "rul_error",
        "alert_30", "alert_urgency", "cost_at_risk_usd",
        "model_version", "predicted_at", "gold_subset"
    ]
    return spark.createDataFrame(pdf[gold_cols])

print("Generating Gold predictions for FD001...")
gold_fd001 = generate_gold(silver_fd001, "FD001")

print("Generating Gold predictions for FD003...")
gold_fd003 = generate_gold(silver_fd003, "FD003")

# Union both subsets
gold_all = gold_fd001.union(gold_fd003)

# Write Gold Delta table
(gold_all.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.predictive_maintenance.gold_predictions"))

print("\n✅ Gold Delta table written: workspace.predictive_maintenance.gold_predictions")
print(f"   Total rows: {gold_all.count():,}")

# Business summary
gold_pd = gold_all.toPandas()
alert_engines = gold_pd[gold_pd["alert_30"] == 1]["unit_id"].nunique()
total_cost    = gold_pd["cost_at_risk_usd"].sum()
critical      = (gold_pd["alert_urgency"] == "CRITICAL").sum()
warning       = (gold_pd["alert_urgency"] == "WARNING").sum()

print("\n=== BUSINESS SUMMARY ===")
print(f"  Engines in alert window (RUL ≤ 30):  {alert_engines}")
print(f"  CRITICAL observations (RUL ≤ 10):    {critical:,}")
print(f"  WARNING observations  (RUL ≤ 30):    {warning:,}")
print(f"  Total cost at risk (USD):             ${total_cost:,.0f}")
print(f"\n  Urgency distribution:")
print(gold_pd["alert_urgency"].value_counts().to_string())

Generating Gold predictions for FD001...


/home/spark-8fbc7758-30ef-4801-8788-f8/.ipykernel/80/command-4707199246831142-2666413901:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  pdf["predicted_at"]     = datetime.utcnow().isoformat()


Generating Gold predictions for FD003...


/home/spark-8fbc7758-30ef-4801-8788-f8/.ipykernel/80/command-4707199246831142-2666413901:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  pdf["predicted_at"]     = datetime.utcnow().isoformat()



✅ Gold Delta table written: workspace.predictive_maintenance.gold_predictions
   Total rows: 75,043

=== BUSINESS SUMMARY ===
  Engines in alert window (RUL ≤ 30):  96
  CRITICAL observations (RUL ≤ 10):    1,179
  WARNING observations  (RUL ≤ 30):    1,614
  Total cost at risk (USD):             $475,017,834

  Urgency distribution:
alert_urgency
HEALTHY     67443
MONITOR      4807
WARNING      1614
CRITICAL     1179


In [0]:
# ============================================================
# CELL 3 — Register Ridge Model with Signature (UC required)
# ============================================================

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

EXPERIMENT_PATH = "/Users/gkolthoff1@live.maryville.edu/rul_prediction_cmapss"
model_name      = "ridge_rul_predictor_cmapss"

# Load Silver test data to create signature sample
silver_fd001 = spark.table("workspace.predictive_maintenance.silver_fd001")
feature_cols = (
    [f"sensor_{s}_norm" for s in [2,3,4,7,8,9,11,12,13,15,17,20,21]] +
    [f"sensor_{s}_roll_mean" for s in [2,3,4,7,8,9,11,12,13,15,17,20,21]] +
    [f"sensor_{s}_roll_std" for s in [2,3,4,7,8,9,11,12,13,15,17,20,21]]
)

sample_pd = silver_fd001.filter("split = 'test'").limit(100).toPandas()
X_sample  = sample_pd[feature_cols].fillna(0)
y_sample  = ridge_model.predict(X_sample)

# Infer signature from sample
signature = infer_signature(X_sample, y_sample)
input_example = X_sample.iloc[:5]

# Re-log model with signature in a new run
mlflow.set_experiment(EXPERIMENT_PATH)

with mlflow.start_run(run_name="ridge_production_registered"):
    mlflow.log_param("model_type", "RidgeCV_production")
    mlflow.log_param("best_alpha", 100.0)
    mlflow.log_metric("test_rmse", 23.88)
    mlflow.log_metric("test_mae", 17.24)
    mlflow.log_metric("alert_recall", 0.152)
    mlflow.log_metric("alert_f1", 0.260)

    mlflow.sklearn.log_model(
        ridge_model,
        name="ridge_model_signed",
        signature=signature,
        input_example=input_example,
        registered_model_name=model_name
    )
    new_run_id = mlflow.active_run().info.run_id

print(f"✅ Model registered with signature: {model_name}")
print(f"   Run ID: {new_run_id}")
print(f"   Test RMSE: 23.88 cycles | Alert F1: 0.260")
print(f"\nView in MLflow UI → Models → {model_name}")

🔗 View Logged Model at: https://dbc-875e7f65-c010.cloud.databricks.com/ml/experiments/3021995221306764/models/m-32ce3906418c47b89170a9f8b8dc4dd8?o=7474646022806704
Registered model 'ridge_rul_predictor_cmapss' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.ridge_rul_predictor_cmapss': https://dbc-875e7f65-c010.cloud.databricks.com/explore/data/models/workspace/default/ridge_rul_predictor_cmapss/version/1?o=7474646022806704


✅ Model registered with signature: ridge_rul_predictor_cmapss
   Run ID: ac80ea092925443ebe67a413ae2b3e47
   Test RMSE: 23.88 cycles | Alert F1: 0.260

View in MLflow UI → Models → ridge_rul_predictor_cmapss


In [0]:
# ============================================================
# CELL 4 (FIXED) — Snowflake write using executemany instead
# ============================================================

import snowflake.connector
import numpy as np

SF_ACCOUNT   = "omc31387.us-east-1"
SF_USER      = "GKOLT18DEV"
SF_PASSWORD  = "Q7nKFok$q2*D%QY"
SF_DATABASE  = "PREDICTIVE_MAINTENANCE"
SF_SCHEMA    = "GOLD"
SF_WAREHOUSE = "COMPUTE_WH"
SF_TABLE     = "GOLD_PREDICTIONS"

gold_pd = spark.table("workspace.predictive_maintenance.gold_predictions").toPandas()
print(f"Loaded {len(gold_pd):,} rows")

conn = snowflake.connector.connect(
    account=SF_ACCOUNT, user=SF_USER, password=SF_PASSWORD,
    warehouse=SF_WAREHOUSE, database=SF_DATABASE, schema=SF_SCHEMA
)
cursor = conn.cursor()

cursor.execute(f"""
CREATE OR REPLACE TABLE {SF_TABLE} (
    UNIT_ID INTEGER, TIME_CYCLES INTEGER, SPLIT VARCHAR(10),
    SUBSET VARCHAR(10), RUL_ACTUAL FLOAT, RUL_PREDICTED FLOAT,
    RUL_ERROR FLOAT, ALERT_30 INTEGER, ALERT_URGENCY VARCHAR(20),
    COST_AT_RISK_USD FLOAT, MODEL_VERSION VARCHAR(50),
    PREDICTED_AT VARCHAR(50), GOLD_SUBSET VARCHAR(10)
)
""")
print("✅ Table created")

# Convert to list of tuples for executemany
gold_pd = gold_pd.replace({np.nan: None})
rows = list(gold_pd.itertuples(index=False, name=None))

BATCH = 5000
total = 0
for i in range(0, len(rows), BATCH):
    batch = rows[i:i+BATCH]
    cursor.executemany(
        f"INSERT INTO {SF_TABLE} VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
        batch
    )
    total += len(batch)
    print(f"  Inserted {total:,}/{len(rows):,} rows...")

cursor.execute(f"SELECT COUNT(*) FROM {SF_TABLE}")
print(f"✅ Verified: {cursor.fetchone()[0]:,} rows in Snowflake")

cursor.close()
conn.close()

Loaded 75,043 rows
✅ Table created
  Inserted 5,000/75,043 rows...
  Inserted 10,000/75,043 rows...
  Inserted 15,000/75,043 rows...
  Inserted 20,000/75,043 rows...
  Inserted 25,000/75,043 rows...
  Inserted 30,000/75,043 rows...
  Inserted 35,000/75,043 rows...
  Inserted 40,000/75,043 rows...
  Inserted 45,000/75,043 rows...
  Inserted 50,000/75,043 rows...
  Inserted 55,000/75,043 rows...
  Inserted 60,000/75,043 rows...
  Inserted 65,000/75,043 rows...
  Inserted 70,000/75,043 rows...
  Inserted 75,000/75,043 rows...
  Inserted 75,043/75,043 rows...
✅ Verified: 75,043 rows in Snowflake
